In [1]:
import pandas as pd
from pathlib import Path

in_dir = Path("../data/by_participant_standardized")   # adjust if needed

summary = []

for csv_path in sorted(in_dir.glob("participant_*.csv")):
    df = pd.read_csv(csv_path, low_memory=False)

    if "trial" not in df.columns:
        print(f"{csv_path.name}: ❌ no 'trial' column found")
        continue

    unique_trials = df["trial"].dropna().unique()
    n_trials = len(unique_trials)

    participant_id = csv_path.stem.split("_")[-1]

    status = "OK" if n_trials == 19 else "MISMATCH"
    summary.append((participant_id, n_trials, sorted(unique_trials), status))

# Print summary
print("\n=== Trial Count Check ===")
for pid, n, trials, status in summary:
    print(f"Participant {pid}: {n} trials → {status}")

# Optional: highlight mismatches explicitly
mismatches = [s for s in summary if s[3] == "MISMATCH"]
if mismatches:
    print("\n❗ Participants with missing or extra trials:")
    for pid, n, trials, _ in mismatches:
        print(f" - Participant {pid}: {n} trials ({trials})")
else:
    print("\n✔ All participants have exactly 19 trials.")


=== Trial Count Check ===
Participant 1: 19 trials → OK
Participant 10: 19 trials → OK
Participant 11: 20 trials → MISMATCH
Participant 13: 19 trials → OK
Participant 14: 19 trials → OK
Participant 15: 34 trials → MISMATCH
Participant 17: 34 trials → MISMATCH
Participant 18: 34 trials → MISMATCH
Participant 19: 34 trials → MISMATCH
Participant 2: 19 trials → OK
Participant 20: 34 trials → MISMATCH
Participant 21: 34 trials → MISMATCH
Participant 22: 33 trials → MISMATCH
Participant 23: 34 trials → MISMATCH
Participant 24: 33 trials → MISMATCH
Participant 25: 19 trials → OK
Participant 26: 19 trials → OK
Participant 27: 34 trials → MISMATCH
Participant 28: 20 trials → MISMATCH
Participant 29: 33 trials → MISMATCH
Participant 3: 19 trials → OK
Participant 30: 33 trials → MISMATCH
Participant 31: 33 trials → MISMATCH
Participant 32: 19 trials → OK
Participant 33: 20 trials → MISMATCH
Participant 34: 19 trials → OK
Participant 35: 19 trials → OK
Participant 36: 33 trials → MISMATCH
Partic

Trying to find the 'stimulus' column in each participant file and count unique stimuli.

In [ ]:
import pandas as pd
from pathlib import Path

in_dir = Path("../data/by_participant_standardized")   # adjust if needed

summary = []

for csv_path in sorted(in_dir.glob("participant_*.csv")):
    df = pd.read_csv(csv_path, low_memory=False)

    if "stimulus" not in df.columns:
        print(f"{csv_path.name}: ❌ no 'stimulus' column found")
        continue

    unique_stimuli = df["stimulus"].dropna().unique()
    n_stimuli = len(unique_stimuli)

    participant_id = csv_path.stem.split("_")[-1]

    status = "OK" if n_stimuli == 19 else "MISMATCH"
    summary.append((participant_id, n_stimuli, sorted(unique_stimuli), status))

# Print summary
print("\n=== Stimulus Count Check ===")
for pid, n, stimuli, status in summary:
    print(f"Participant {pid}: {n} stimuli → {status}")

# Optional: highlight mismatches explicitly
mismatches = [s for s in summary if s[3] == "MISMATCH"]
if mismatches:
    print("\n❗ Participants with missing or extra stimuli:")
    for pid, n, stimuli, _ in mismatches:
        print(f" - Participant {pid}: {n} stimuli ({stimuli})")
else:
    print("\n✔ All participants have exactly 19 stimuli.")

extracting the 'trial' column from each participant file, tryping to count unique trials

In [6]:
import pandas as pd
from pathlib import Path

in_dir = Path("../data/by_participant_standardized")

all_stimuli = set()

for csv_path in in_dir.glob("participant_*.csv"):
    df = pd.read_csv(csv_path, low_memory=False)
    if "content" not in df.columns:
        continue

    all_stimuli.update(df["content"].dropna().unique())

print("Unique stimuli:", len(all_stimuli))
for stim in sorted(all_stimuli):
    print(stim)

Unique stimuli: 109
-
0a574db0baf466c1a090ffa8652335d1_1280x1024.jpg
1 coucou d.jpg
1 coucou g.jpg
10 devant.jpg
11 yeux chat d.jpg
11 yeux chat gauche.jpg
12 tete chat droite.jpg
12 tete chat gauche.jpg
13 tete pointage chat droite.jpg
13 tete pointage chat gauche.jpg
14 devant point chat droite.jpg
14 devant point chat gauche.jpg
15 devant - copie.jpg
15 devant.jpg
16 voc chat gauche.jpg
16 voc droite chat.jpg
17 voc devant d.jpg
17 voc devant.jpg
18 au revoir.jpg
18 aurevoir.jpg
199d60f088b758400ff70edc242a7b64.avi
1a60d6352dd0a48a32f3258bda3c32c1_1280x1024.jpg
1coucou g.jpg
1f04c7a7f5de1603711984ba08dcb346_1280x1024.jpg
2 devant.jpg
20ea2ccd488a6ceae3bf695a1ed8459a_1280x1024.jpg
23d4c4c9d202d05be1c4bb598d20e84d.avi
2dc4d831e2078e304cf469b9c7827fbb_1280x1024.jpg
3 regard chien d.jpg
3 regard chien g.jpg
4 tete chien d.jpg
4 tete chien g.jpg
41ae6f2ed847c4b059ebd02f4aaf3f72_1280x1024.jpg
4a42343bf6a344d6603b41701f95dcc0_1280x1024.jpg
5 tete point chien d.jpg
5 tete point chien g.jpg


## Stimulus normalization

In [9]:
import pandas as pd

def normalize_stimulus(stimulus_name: str) -> str:
    if pd.isna(stimulus_name):
        return "other"
    
    n = str(stimulus_name).strip().lower()

    # trivial / empty
    if n in {"", "-", "none"}:
        return "other"
    if "noimage" in n:
        return "noimage"

    # coucou faces (left/right greeting)
    if "coucou" in n:
        if " g" in n or " gauche" in n:
            return "coucou_left"
        if " d" in n or " droite" in n:
            return "coucou_right"
        return "coucou"

    # dog stimuli (chien)
    if "chien" in n:
        if "regard" in n:  # gaze
            if " g" in n or " gauche" in n:
                return "dog_gaze_left"
            if " d" in n or " droite" in n:
                return "dog_gaze_right"
            return "dog_gaze"
        if "tete point" in n:  # pointing with head
            if " g" in n or " gauche" in n:
                return "dog_point_left"
            if " d" in n or " droite" in n:
                return "dog_point_right"
            return "dog_point"
        if "devant point" in n:
            if " g" in n or " gauche" in n:
                return "dog_point_left"
            if " d" in n or " droite" in n:
                return "dog_point_right"
            return "dog_point"
        if "tete" in n:  # head only
            if " g" in n or " gauche" in n:
                return "dog_head_left"
            if " d" in n or " droite" in n:
                return "dog_head_right"
            return "dog_head"
        if "devant" in n:
            return "dog_front"
        if "voc" in n:
            if " g" in n or " gauche" in n:
                return "dog_voc_left"
            if " d" in n or " droite" in n:
                return "dog_voc_right"
            return "dog_voc"
        return "dog_other"

    # cat stimuli (chat)
    if "chat" in n:
        if "yeux" in n:
            if " g" in n or " gauche" in n:
                return "cat_eyes_left"
            if " d" in n or " droite" in n:
                return "cat_eyes_right"
            return "cat_eyes"
        if "tete pointage" in n:
            if " g" in n or " gauche" in n:
                return "cat_point_left"
            if " d" in n or " droite" in n:
                return "cat_point_right"
            return "cat_point"
        if "devant point" in n:
            if " g" in n or " gauche" in n:
                return "cat_point_left"
            if " d" in n or " droite" in n:
                return "cat_point_right"
            return "cat_point"
        if "tete" in n:
            if " g" in n or " gauche" in n:
                return "cat_head_left"
            if " d" in n or " droite" in n:
                return "cat_head_right"
            return "cat_head"
        if "voc" in n:
            if " gauche" in n or " g " in n:
                return "cat_voc_left"
            if " droite" in n or " d " in n:
                return "cat_voc_right"
            if "devant" in n:
                return "cat_voc_front"
            return "cat_voc"
        if "devant" in n:
            return "cat_front"
        return "cat_other"

    # generic "devant" without chien/chat (likely child front image)
    if "devant" in n:
        return "front_view"

    # au revoir / aurevoir
    if "au revoir" in n or "aurevoir" in n:
        return "goodbye_face"

    # emotion video families (triste/joie stuff)
    if "bonbons" in n and "triste" in n and "joie" in n:
        return "emo_bonbons"
    if "vole" in n and "triste" in n and "joie" in n:
        return "emo_vole"
    if "punition orale" in n and "triste" in n and "joie" in n:
        return "emo_punition"
    if "sous l'eau" in n and "joie" in n and "triste" in n:
        return "emo_sous_leau"
    if "tombe joie vs triste" in n or ("tombe" in n and "joie" in n and "triste" in n):
        return "emo_tombe"
    if "cadeau dernier" in n:
        return "emo_cadeau"

    # A/B pose variants with joy/sad labels but no specific family word
    if ("triste" in n and "joie" in n) or ("joie" in n and "triste" in n):
        return "emo_faces_ab"

    # neutral faces
    if "neutre visage gris" in n:
        return "neutral_face"
    if n.startswith("neutre") and n.endswith(".avi"):
        return "neutral_video"

    # balloon eye tracking
    if "eye tracking" in n and "ballon" in n:
        if "droite" in n:
            return "ballon_right"
        if "gauche" in n:
            return "ballon_left"
        return "ballon"

    # federica / fede videos
    if "fede" in n or "federica" in n:
        return "federica"

    # vnvd / vnvg
    if "vnvd" in n or "vnvg" in n:
        return "vnv_misc"

    # hashed / generic video files (no meaningful tokens)
    if n.endswith(".avi"):
        return "other_video"

    # hashed / generic still frames
    if n.endswith(".jpg") or n.endswith(".png"):
        if "_1280x1024" in n:
            return "other_frame"
        return "other_image"

    # fallback
    return "other"

## Apply mapping to all participant files & show counts per category


In [10]:
from pathlib import Path

in_dir = Path("../data/by_participant_standardized")   # adjust if needed
out_dir = Path("../data/by_participant_with_stim")
out_dir.mkdir(exist_ok=True)

all_counts = []

for csv_path in sorted(in_dir.glob("participant_*.csv")):
    df = pd.read_csv(csv_path, low_memory=False)

    # Find stimulus column (case-insensitive)
    cols_lower = {c.lower(): c for c in df.columns}
    stim_col = cols_lower.get("stimulus")
    if stim_col is None:
        print(f"{csv_path.name}: no 'stimulus' column, skipping")
        continue

    # apply normalization on stimulus
    df["stim_category"] = df[stim_col].apply(normalize_stimulus)

    # save updated file
    df.to_csv(out_dir / csv_path.name, index=False)

    # counts per category for this participant
    pid = csv_path.stem.split("_")[-1]
    counts = (df["stim_category"]
              .value_counts()
              .rename_axis("stim_category")
              .reset_index(name="n_samples"))
    counts["participant"] = pid
    all_counts.append(counts)

# combine across participants
if all_counts:
    counts_df = pd.concat(all_counts, ignore_index=True)

    # total samples per category across all participants
    total_counts = (counts_df.groupby("stim_category")["n_samples"]
                               .sum()
                               .sort_values(ascending=False))
    print("\n=== Total samples per stimulus category (all participants) ===")
    print(total_counts)

    # number of participants with at least one sample per category
    participant_coverage = (counts_df.groupby("stim_category")["participant"]
                                      .nunique()
                                      .sort_values(ascending=False))
    print("\n=== Number of participants per category ===")
    print(participant_coverage)


=== Total samples per stimulus category (all participants) ===
stim_category
vnv_misc           254664
noimage            157523
federica           136991
ballon_right       125447
front_view          88521
ballon_left         83713
emo_faces_ab        64522
emo_vole            37566
emo_sous_leau       34399
other_video         30305
emo_punition        29867
emo_tombe           27914
emo_cadeau          26751
emo_bonbons         24269
dog_point_right     24026
cat_point_left      23272
coucou_left         17861
goodbye_face        16800
dog_gaze_right      12053
cat_eyes_left       12047
dog_head_right      12005
coucou_right        11944
dog_voc_right       11855
cat_head_left       11679
neutral_face        10861
dog_point_left      10402
cat_point_right     10254
cat_voc_left         5784
cat_voc              5718
dog_gaze_left        5516
dog_voc_left         5508
cat_eyes_right       5499
cat_head_right       5303
dog_head_left        5233
cat_voc_right        5085
Name: n_samp

checking the postition  of noimage to determine if this is at the end of a trial

In [16]:
import pandas as pd
from pathlib import Path

dir_with_stim = Path("../data/by_participant_with_stim")

for csv_path in dir_with_stim.glob("participant_28.csv"):
    df = pd.read_csv(csv_path)

    if "stim_category" not in df.columns:
        continue

    noimg_rows = df[df["stim_category"] == "noimage"]

    if noimg_rows.empty:
        continue

    pid = csv_path.stem.split("_")[-1]
    print(f"\nParticipant {pid}:")

    # where in trials does noimage appear?
    positions = noimg_rows.groupby("trial").apply(
        lambda g: (g.index.min(), g.index.max())
    )
    print("Trial → (first_row_index, last_row_index) for noimage")
    print(positions)


Participant 28:
Trial → (first_row_index, last_row_index) for noimage
trial
Trial001     (8522, 8617)
Trial002     (8618, 8707)
Trial003     (8708, 8818)
Trial004     (8819, 8874)
Trial005     (8875, 8941)
Trial006     (8942, 9006)
Trial007     (9007, 9089)
Trial008     (9090, 9210)
Trial009     (9211, 9308)
Trial010     (9309, 9374)
Trial011     (9375, 9436)
Trial012     (9437, 9502)
Trial013     (9503, 9569)
Trial014     (9570, 9636)
Trial015     (9637, 9703)
Trial016     (9704, 9770)
Trial017     (9771, 9836)
Trial018     (9837, 9903)
Trial019    (9904, 11224)
dtype: object


/var/folders/g3/x3j647bn42j_bpmd_9vl047w0000gn/T/ipykernel_67260/2764162358.py:7: DtypeWarning: Columns (14,15,16,17,18,19,20,21,22,23,24,25,26,27,29,30,31,32,33,34,35,36,37,38,39,40,41,42) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)
/var/folders/g3/x3j647bn42j_bpmd_9vl047w0000gn/T/ipykernel_67260/2764162358.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  positions = noimg_rows.groupby("trial").apply(


functtion to remove noimage from the end of trials

In [ ]:
import pandas as pd
from pathlib import Path

# ---------- 1) Function to trim noimage tails per trial ----------

def trim_noimage_tails(df: pd.DataFrame) -> pd.DataFrame:
    """
    For each trial, keep only the rows from the first non-noimage
    stimulus until the last non-noimage stimulus.
    Removes the blank-screen tail (and any leading noimage) per trial.
    """
    if "trial" not in df.columns or "stim_category" not in df.columns:
        raise ValueError("trial or stim_category column missing.")

    trimmed_parts = []

    # (assumes df is already sorted by time within trial)
    for trial_id, tdf in df.groupby("trial", sort=False):
        # rows that actually contain a real stimulus (not noimage)
        mask_real = tdf["stim_category"] != "noimage"

        # if the entire trial is noimage → skip this trial completely
        if not mask_real.any():
            continue

        # index labels of first and last non-noimage rows
        first_label = mask_real.idxmax()
        last_label = mask_real[::-1].idxmax()

        # slice on the original df, not on tdf (to preserve structure)
        trimmed_parts.append(df.loc[first_label:last_label])

    if not trimmed_parts:
        # nothing left, return empty frame with same columns
        return df.iloc[0:0].copy()

    trimmed = pd.concat(trimmed_parts, ignore_index=True)
    return trimmed


# ---------- 2) Apply to all participants ----------

in_dir = Path("../data/by_participant_with_stim")   # input: with stim_category & trial
out_dir = Path("../data/by_participant_trimmed")    # output: trimmed trials
out_dir.mkdir(parents=True, exist_ok=True)

summary = []

for csv_path in sorted(in_dir.glob("participant_*.csv")):
    df = pd.read_csv(csv_path, low_memory=False)

    if "trial" not in df.columns or "stim_category" not in df.columns:
        print(f"Skipping {csv_path.name}: missing 'trial' or 'stim_category'")
        continue

    n_before = len(df)
    n_noimage_before = (df["stim_category"] == "noimage").sum()

    df_trimmed = trim_noimage_tails(df)

    n_after = len(df_trimmed)
    n_noimage_after = (df_trimmed["stim_category"] == "noimage").sum()

    # save trimmed file
    out_path = out_dir / csv_path.name
    df_trimmed.to_csv(out_path, index=False)

    pid = csv_path.stem.split("_")[-1]
    summary.append((pid, n_before, n_after, n_noimage_before, n_noimage_after))

    print(
        f"{csv_path.name}: rows {n_before} → {n_after} "
        f"(noimage {n_noimage_before} → {n_noimage_after})"
    )

# ---------- 3) compact summary ----------

print("\n=== Trimming summary (per participant) ===")
for pid, n_before, n_after, n_noimg_b, n_noimg_a in summary:
    print(
        f"Participant {pid}: rows {n_before} → {n_after}, "
        f"noimage {n_noimg_b} → {n_noimg_a}"
    )

participant_1.csv: rows 8531 → 41734 (noimage 0 → 0)
participant_10.csv: rows 9289 → 45772 (noimage 0 → 0)
participant_11.csv: rows 16297 → 94060 (noimage 0 → 0)
participant_13.csv: rows 13231 → 149411 (noimage 3849 → 69282)
participant_14.csv: rows 13087 → 148289 (noimage 3948 → 71064)
participant_15.csv: rows 19503 → 223915 (noimage 0 → 0)
participant_17.csv: rows 43963 → 707273 (noimage 11492 → 206856)
participant_18.csv: rows 32940 → 464239 (noimage 11974 → 215532)
participant_19.csv: rows 34307 → 482494 (noimage 12604 → 226872)
participant_2.csv: rows 7132 → 34860 (noimage 0 → 0)
participant_20.csv: rows 34502 → 487302 (noimage 12877 → 231786)
participant_21.csv: rows 24316 → 336625 (noimage 8903 → 160254)
participant_22.csv: rows 18517 → 18517 (noimage 0 → 0)
participant_23.csv: rows 21750 → 227774 (noimage 4267 → 4267)
participant_24.csv: rows 36853 → 62704 (noimage 0 → 0)
participant_25.csv: rows 17812 → 20584 (noimage 0 → 0)
participant_26.csv: rows 17768 → 20528 (noimage 0 → 